 # Medium BIT-S4-DD8 training and HPO



 The medium BIT architecture is implemented in `src/satchange/models/bit.py`. This notebook does not redefine it: `create_change_model()` selects `BIT` from the model factory and passes the parameters from `configs/bit.yaml`.



 A fresh **Restart Kernel → Run All** is safe. Dataset preparation, full training, HPO, and tiny-overfit diagnostics run only when their explicit flags are enabled.

In [1]:
# %%
import json
from pathlib import Path
from uuid import uuid4

import torch
from torch.utils.data import DataLoader, Subset

from satchange import BCEDiceLoss, fit_binary_change_model
from satchange.data import (
    RandomFlipRotate,
    create_dataset_dataloader,
    prepare_levir_cd,
)
from satchange.models import count_model_parameters, create_change_model
from satchange.optimization import run_parameter_search
from satchange.utils import load_config


In [2]:
# CONFIG, PROJECT_ROOT etc.


current_directory = Path.cwd().resolve()
project_root = (
    current_directory.parent
    if current_directory.name == "notebooks"
    else current_directory
)
if not (project_root / "pyproject.toml").is_file():
    raise FileNotFoundError(
        "Run this notebook from the project root or notebooks directory."
    )

config_path = project_root / "configs" / "bit_full.yaml"
config = load_config(config_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(config["project"]["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config["project"]["seed"])

model = create_change_model(
    model_name=config["model"]["name"],
    parameters=config["model"]["parameters"],
).to(device)

print("Config:", config_path)
print("Device:", device)
print("Created model:", type(model).__name__)
print("Architecture parameters:", config["model"]["parameters"])
print("Parameter counts:", count_model_parameters(model))


Config: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/configs/bit_full.yaml
Device: cuda
Created model: BIT
Architecture parameters: {'in_channels': 3, 'out_channels': 1, 'embedding_dim': 32, 'token_length': 4, 'num_heads': 8, 'encoder_head_dim': 64, 'decoder_head_dim': 64, 'encoder_depth': 1, 'decoder_depth': 8, 'dropout': 0.0, 'pretrained_backbone': False}
Parameter counts: {'total_parameters': 3495973, 'trainable_parameters': 3495973}


In [3]:
image_size = config["data"]["image_size"]
before = torch.rand(1, 3, image_size, image_size, device=device)
after = torch.rand(1, 3, image_size, image_size, device=device)
model.eval()
with torch.no_grad():
    logits = model(before, after)

expected_shape = (1, 1, image_size, image_size)
assert tuple(logits.shape) == expected_shape
assert torch.isfinite(logits).all()
print("Forward output:", logits.shape)


Forward output: torch.Size([1, 1, 1024, 1024])


 ## Dataset location and optional preparation



 The training and HPO blocks use `dataset_root` below. Enable `RUN_DATA_SETUP` only when LEVIR-CD must be downloaded, extracted, or validated.

In [4]:
# DATA

RUN_DATA_SETUP = False

dataset_root = project_root / config["dataset"]["prepared_directory"]
if RUN_DATA_SETUP:
    dataset_root = prepare_levir_cd(
        config=config,
        project_root=project_root,
    )
    print("Dataset ready:", dataset_root)
else:
    print("Dataset preparation skipped. Expected location:", dataset_root)


Dataset preparation skipped. Expected location: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD


In [5]:
THRESHOLDS = [round(value / 100, 2) for value in range(5, 96, 5)]
loss_config = config["training"]["loss"]

print("Thresholds:", THRESHOLDS)
print("Loss parameters:", loss_config)


Thresholds: [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
Loss parameters: {'bce_weight': 0.3, 'dice_weight': 0.7, 'pos_weight': 10}


 ## Full medium BIT training



 Set `RUN_FULL_TRAINING = True` to train one medium BIT configuration. Each new run creates a fresh model, loss, optimizer, and loaders. Set `RESUME_FROM` to that run's `last.pt` to continue it; `NUMBER_OF_EPOCHS` then means additional epochs.

In [15]:
# RUN_FULL_TRAINING = False
# RUN_NAME = "levir_bit_medium_baseline_v1"
# NUMBER_OF_EPOCHS = config["training"]["epochs"]
# TRAINING_PRECISION = "fp32"
# RESUME_FROM = None
# TRAINING_OUTPUT_ROOT = project_root / "artifacts" / "training_runs"

# training_history = []

# if RUN_FULL_TRAINING:
#     if not dataset_root.is_dir():
#         raise FileNotFoundError(
#             f"LEVIR-CD was not found at {dataset_root}. "
#             "Run the dataset-preparation cell first."
#         )

#     train_transform = RandomFlipRotate(
#         horizontal_flip_probability=0.5,
#         vertical_flip_probability=0.5,
#     )
#     _, train_loader = create_dataset_dataloader(
#         dataset_root=dataset_root,
#         split="train",
#         image_size=config["data"]["image_size"],
#         batch_size=config["data"]["batch_size"],
#         num_workers=config["data"]["num_workers"],
#         transform=train_transform,
#         shuffle=True,
#         pin_memory=torch.cuda.is_available(),
#         seed=config["project"]["seed"],
#     )
#     _, validation_loader = create_dataset_dataloader(
#         dataset_root=dataset_root,
#         split="val",
#         image_size=config["data"]["image_size"],
#         batch_size=config["data"]["batch_size"],
#         num_workers=config["data"]["num_workers"],
#         pin_memory=torch.cuda.is_available(),
#     )

#     training_model = create_change_model(
#         model_name=config["model"]["name"],
#         parameters=config["model"]["parameters"],
#     ).to(device)
#     training_loss = BCEDiceLoss(
#         bce_weight=loss_config["bce_weight"],
#         dice_weight=loss_config["dice_weight"],
#         pos_weight=loss_config["pos_weight"],
#     ).to(device)
#     training_optimizer = torch.optim.AdamW(
#         training_model.parameters(),
#         lr=config["training"]["learning_rate"],
#         weight_decay=config["training"]["weight_decay"],
#     )

#     training_history = fit_binary_change_model(
#         model=training_model,
#         train_loader=train_loader,
#         validation_loader=validation_loader,
#         loss_function=training_loss,
#         optimizer=training_optimizer,
#         device=device,
#         number_of_epochs=NUMBER_OF_EPOCHS,
#         thresholds=THRESHOLDS,
#         output_root=TRAINING_OUTPUT_ROOT,
#         model_name=config["model"]["name"],
#         run_name=RUN_NAME,
#         checkpoint_interval=5,
#         config=config,
#         resume_from=RESUME_FROM,
#         scheduler=None,
#         precision=TRAINING_PRECISION,
#     )

#     best_training_result = max(
#         training_history,
#         key=lambda result: result["validation_f1"],
#     )
#     print("Best training result:", best_training_result)
# else:
#     print("Full training skipped; set RUN_FULL_TRAINING=True to enable it.")


 ## Medium BIT hyperparameter search



 Top-level keys such as `weight_decay` tune training. Architecture keys use the `model__` prefix and are moved into a fresh nested `model_parameters` dictionary for each trial. For example: `model__embedding_dim`, `model__token_length`, or `model__decoder_head_dim`.

In [8]:
# ============================================================
# SAVE COMPLETED STAGE 2 RESULT
# ============================================================

BEST_MODEL_PARAMETERS = {
    "in_channels": 3,
    "out_channels": 1,
    "embedding_dim": 64,
    "token_length": 4,
    "num_heads": 8,
    "encoder_head_dim": 64,
    "decoder_head_dim": 64,
    "encoder_depth": 1,
    "decoder_depth": 8,
    "dropout": 0.0,
    "pretrained_backbone": False,
}

STAGE2_SUMMARY = {
    "candidate_a": {
        "f1": [
            0.843443,
            0.830667,
            0.789656,
        ],
        "mean_f1": 0.8212553333333333,
    },

    "candidate_b": {
        "f1": [
            0.839830,
            0.843332830567492,
            0.8349696982168033,
        ],
        "mean_f1": 0.839377509594765,
    },

    "best_model_parameters": BEST_MODEL_PARAMETERS,
}

STAGE2_SUMMARY_PATH = (
    project_root
    / "artifacts"
    / "hyperparameter_search"
    / "bit_full_stage2_architecture_selection.json"
)

STAGE2_SUMMARY_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with STAGE2_SUMMARY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        STAGE2_SUMMARY,
        file,
        indent=2,
    )

print("Saved:", STAGE2_SUMMARY_PATH)
print("BEST_MODEL_PARAMETERS:", BEST_MODEL_PARAMETERS)

Saved: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/bit_full_stage2_architecture_selection.json
BEST_MODEL_PARAMETERS: {'in_channels': 3, 'out_channels': 1, 'embedding_dim': 64, 'token_length': 4, 'num_heads': 8, 'encoder_head_dim': 64, 'decoder_head_dim': 64, 'encoder_depth': 1, 'decoder_depth': 8, 'dropout': 0.0, 'pretrained_backbone': False}


In [9]:
HPO_EPOCHS = 40
VERIFY_EPOCHS = 60
FINAL_EPOCHS = 100

FULL_BATCH_SIZE = 4

HPO_GPU_IDS = [0, 1, 2, 3]

HPO_OUTPUT_ROOT = (
    project_root
    / "artifacts"
    / "hyperparameter_search"
)


print(
    "Loaded BEST_MODEL_PARAMETERS:",
    BEST_MODEL_PARAMETERS,
)

Loaded BEST_MODEL_PARAMETERS: {'in_channels': 3, 'out_channels': 1, 'embedding_dim': 64, 'token_length': 4, 'num_heads': 8, 'encoder_head_dim': 64, 'decoder_head_dim': 64, 'encoder_depth': 1, 'decoder_depth': 8, 'dropout': 0.0, 'pretrained_backbone': False}


In [18]:
# ============================================================
# STAGE 3A — LOSS SCREENING 0.3 / 0.7
# ============================================================

SEARCH_SPACE = {
    "positive_weight": [
        5,
        10,
        15,
        20,
        25,
        30,
    ],
}

FIXED_PARAMETERS = {
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "scheduler": "none",

    "bce_weight": 0.3,
    "dice_weight": 0.7,

    "batch_size": FULL_BATCH_SIZE,
    "image_size": config["data"]["image_size"],
    "num_workers": config["data"]["num_workers"],
    "seed": 42,
    "precision": "fp32",

    "model_parameters": BEST_MODEL_PARAMETERS,
}

SEARCH_NAME = (
    "bit_full_stage3_loss_"
    "bce03_dice07_40ep_seed42_v2"
)

stage3a_results = run_parameter_search(
    search_space=SEARCH_SPACE,
    fixed_parameters=FIXED_PARAMETERS,
    model_name=config["model"]["name"],
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,
    number_of_epochs=HPO_EPOCHS,
    output_root=HPO_OUTPUT_ROOT,
    search_name=SEARCH_NAME,
    gpu_ids=HPO_GPU_IDS,
    checkpoint_interval=5,
)

Trials: 6
GPUs: [0, 1, 2, 3]

[START] Trial 000 -> GPU 0
seed=42 | lr=0.0003 | wd=0.0001 | pw=5 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=40

[START] Trial 001 -> GPU 1
seed=42 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=40

[START] Trial 002 -> GPU 2
seed=42 | lr=0.0003 | wd=0.0001 | pw=15 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 


[START] Trial 003 -> GPU 3
seed=42 | lr=0.0003 | wd=0.0001 | pw=20 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=40
[Trial 000 | GPU 0] deterministic_algorithms=True | cudnn_deterministic=True | cudnn_benchmark=False
[Trial 003 | GPU 3] deterministic_algorithms=True | cudnn_deterministic=True | cudnn_benchmark=False
[Trial 002 | GPU 2] deterministic_algorithms=True | cudnn_deterministic=True | cudnn_benchmark=False
[Trial 001 | GPU 1] deterministic_algorithms=True | cudnn_deterministic=True | cudnn_benchmark=False

[Trial 003 | GPU 3] Epoch 1/40
Train loss:       0.816992
Validation loss:  0.826261
Best threshold:   0.70
Precision:        0.500245
Recall:           0.488147
F1:               0.494122
IoU:              0.328129

[Trial 001 | GPU 1] Epoch 1/40


In [19]:
# ============================================================
# STAGE 3B — LOSS SCREENING 0.5 / 0.5
# ============================================================

SEARCH_SPACE = {
    "positive_weight": [
        5,
        10,
        15,
        20,
        25,
        30,
    ],
}

FIXED_PARAMETERS = {
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "scheduler": "none",

    "bce_weight": 0.5,
    "dice_weight": 0.5,

    "batch_size": FULL_BATCH_SIZE,
    "image_size": config["data"]["image_size"],
    "num_workers": config["data"]["num_workers"],
    "seed": 42,
    "precision": "fp32",

    "model_parameters": BEST_MODEL_PARAMETERS,
}

SEARCH_NAME = (
    "bit_full_stage3_loss_"
    "bce05_dice05_40ep_seed42_v1"
)

stage3b_results = run_parameter_search(
    search_space=SEARCH_SPACE,
    fixed_parameters=FIXED_PARAMETERS,
    model_name=config["model"]["name"],
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,
    number_of_epochs=HPO_EPOCHS,
    output_root=HPO_OUTPUT_ROOT,
    search_name=SEARCH_NAME,
    gpu_ids=HPO_GPU_IDS,
    checkpoint_interval=5,
)

Trials: 6
GPUs: [0, 1, 2, 3]

[START] Trial 000 -> GPU 0
seed=42 | lr=0.0003 | wd=0.0001 | pw=5 | BCE/Dice=0.5/0.5 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=40

[START] Trial 001 -> GPU 1
seed=42 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.5/0.5 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=40

[START] Trial 002 -> GPU 2
seed=42 | lr=0.0003 | wd=0.0001 | pw=15 | BCE/Dice=0.5/0.5 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 

In [20]:
# ============================================================
# STAGE 3C — SELECT TOP-2 LOSS CONFIGURATIONS
# ============================================================

loss_candidates = []

for result in stage3a_results:
    loss_candidates.append(
        {
            "positive_weight": result["parameters"]["positive_weight"],
            "bce_weight": 0.3,
            "dice_weight": 0.7,
            "seed42_f1": result["best_f1"],
        }
    )

for result in stage3b_results:
    loss_candidates.append(
        {
            "positive_weight": result["parameters"]["positive_weight"],
            "bce_weight": 0.5,
            "dice_weight": 0.5,
            "seed42_f1": result["best_f1"],
        }
    )

TOP_LOSS_CANDIDATES = sorted(
    loss_candidates,
    key=lambda candidate: candidate["seed42_f1"],
    reverse=True,
)[:2]

print("TOP_LOSS_CANDIDATES:")

for candidate in TOP_LOSS_CANDIDATES:
    print(candidate)

TOP_LOSS_CANDIDATES:
{'positive_weight': 20, 'bce_weight': 0.3, 'dice_weight': 0.7, 'seed42_f1': 0.8351435007511724}
{'positive_weight': 10, 'bce_weight': 0.3, 'dice_weight': 0.7, 'seed42_f1': 0.8319510271384011}


In [21]:
loss_multiseed_summary = []


for index, candidate in enumerate(
    TOP_LOSS_CANDIDATES,
    start=1,
):
    SEARCH_SPACE = {
        "seed": [
            42,
            123,
            777,
        ],
    }

    FIXED_PARAMETERS = {
        "learning_rate": 3e-4,
        "weight_decay": 1e-4,
        "scheduler": "none",

        "positive_weight": (
            candidate["positive_weight"]
        ),
        "bce_weight": (
            candidate["bce_weight"]
        ),
        "dice_weight": (
            candidate["dice_weight"]
        ),

        "batch_size": FULL_BATCH_SIZE,
        "image_size": config["data"]["image_size"],
        "num_workers": config["data"]["num_workers"],
        "precision": "fp32",

        "model_parameters": BEST_MODEL_PARAMETERS,
    }

    SEARCH_NAME = (
        f"bit_full_stage3_loss_"
        f"candidate{index}_multiseed_60ep_v2"
    )

    candidate_results = run_parameter_search(
        search_space=SEARCH_SPACE,
        fixed_parameters=FIXED_PARAMETERS,
        model_name=config["model"]["name"],
        dataset_root=dataset_root,
        thresholds=THRESHOLDS,
        number_of_epochs=VERIFY_EPOCHS,
        output_root=HPO_OUTPUT_ROOT,
        search_name=SEARCH_NAME,
        gpu_ids=HPO_GPU_IDS,
        checkpoint_interval=5,
    )

    f1_scores = [
        result["best_f1"]
        for result in candidate_results
    ]

    mean_f1 = (
        sum(f1_scores)
        / len(f1_scores)
    )

    loss_multiseed_summary.append(
        {
            **candidate,
            "mean_f1": mean_f1,
            "scores": f1_scores,
        }
    )

    print(
        candidate,
        "mean F1:",
        mean_f1,
        "scores:",
        f1_scores,
    )


BEST_LOSS = max(
    loss_multiseed_summary,
    key=lambda candidate: candidate["mean_f1"],
)

BEST_POSITIVE_WEIGHT = (
    BEST_LOSS["positive_weight"]
)

BEST_BCE_WEIGHT = (
    BEST_LOSS["bce_weight"]
)

BEST_DICE_WEIGHT = (
    BEST_LOSS["dice_weight"]
)

print()
print("BEST LOSS:", BEST_LOSS)

Trials: 3
GPUs: [0, 1, 2, 3]

[START] Trial 000 -> GPU 0
seed=42 | lr=0.0003 | wd=0.0001 | pw=20 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=60

[START] Trial 001 -> GPU 1
seed=123 | lr=0.0003 | wd=0.0001 | pw=20 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=60

[START] Trial 002 -> GPU 2
seed=777 | lr=0.0003 | wd=0.0001 | pw=20 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth

In [22]:
# ============================================================
# STAGE 4A — WEIGHT DECAY SCREENING
# ============================================================

SEARCH_SPACE = {
    "weight_decay": [
        0.0,
        1e-5,
        1e-4,
        1e-3,
    ],
}

FIXED_PARAMETERS = {
    "learning_rate": 3e-4,
    "scheduler": "none",

    "positive_weight": BEST_POSITIVE_WEIGHT,
    "bce_weight": BEST_BCE_WEIGHT,
    "dice_weight": BEST_DICE_WEIGHT,

    "batch_size": FULL_BATCH_SIZE,
    "image_size": config["data"]["image_size"],
    "num_workers": config["data"]["num_workers"],
    "seed": 42,
    "precision": "fp32",

    "model_parameters": BEST_MODEL_PARAMETERS,
}

SEARCH_NAME = (
    "bit_full_stage4_weight_decay_"
    "40ep_seed42_v1"
)

stage4_seed42_results = run_parameter_search(
    search_space=SEARCH_SPACE,
    fixed_parameters=FIXED_PARAMETERS,
    model_name=config["model"]["name"],
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,
    number_of_epochs=HPO_EPOCHS,
    output_root=HPO_OUTPUT_ROOT,
    search_name=SEARCH_NAME,
    gpu_ids=HPO_GPU_IDS,
    checkpoint_interval=5,
)

TOP_WEIGHT_DECAYS = [
    result["parameters"]["weight_decay"]
    for result in sorted(
        stage4_seed42_results,
        key=lambda result: result["best_f1"],
        reverse=True,
    )[:2]
]

print(
    "TOP_WEIGHT_DECAYS:",
    TOP_WEIGHT_DECAYS,
)

Trials: 4
GPUs: [0, 1, 2, 3]

[START] Trial 000 -> GPU 0
seed=42 | lr=0.0003 | wd=0.0 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=40

[START] Trial 001 -> GPU 1
seed=42 | lr=0.0003 | wd=1e-05 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=40

[START] Trial 002 -> GPU 2
seed=42 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | d

In [23]:
SEARCH_SPACE = {
    "weight_decay": TOP_WEIGHT_DECAYS,
    "seed": [
        42,
        123,
        777,
    ],
}

FIXED_PARAMETERS = {
    "learning_rate": 3e-4,
    "scheduler": "none",

    "positive_weight": BEST_POSITIVE_WEIGHT,
    "bce_weight": BEST_BCE_WEIGHT,
    "dice_weight": BEST_DICE_WEIGHT,

    "batch_size": FULL_BATCH_SIZE,
    "image_size": config["data"]["image_size"],
    "num_workers": config["data"]["num_workers"],
    "precision": "fp32",

    "model_parameters": BEST_MODEL_PARAMETERS,
}

SEARCH_NAME = (
    "bit_full_stage4_weight_decay_"
    "multiseed_60ep_v2"
)

stage4_multiseed_results = run_parameter_search(
    search_space=SEARCH_SPACE,
    fixed_parameters=FIXED_PARAMETERS,
    model_name=config["model"]["name"],
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,
    number_of_epochs=VERIFY_EPOCHS,
    output_root=HPO_OUTPUT_ROOT,
    search_name=SEARCH_NAME,
    gpu_ids=HPO_GPU_IDS,
    checkpoint_interval=5,
)


weight_decay_scores = {
    weight_decay: []
    for weight_decay in TOP_WEIGHT_DECAYS
}


for result in stage4_multiseed_results:
    weight_decay = (
        result["parameters"]["weight_decay"]
    )

    weight_decay_scores[
        weight_decay
    ].append(
        result["best_f1"]
    )


for weight_decay, scores in (
    weight_decay_scores.items()
):
    print(
        weight_decay,
        "mean F1:",
        sum(scores) / len(scores),
        "scores:",
        scores,
    )


BEST_WEIGHT_DECAY = max(
    weight_decay_scores,
    key=lambda weight_decay: (
        sum(weight_decay_scores[weight_decay])
        / len(weight_decay_scores[weight_decay])
    ),
)

print(
    "BEST_WEIGHT_DECAY:",
    BEST_WEIGHT_DECAY,
)

Trials: 6
GPUs: [0, 1, 2, 3]

[START] Trial 000 -> GPU 0
seed=42 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=60

[START] Trial 001 -> GPU 1
seed=123 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=60

[START] Trial 002 -> GPU 2
seed=777 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth

In [24]:
# ============================================================
# STAGE 5A — LEARNING RATE SCREENING
# ============================================================

SEARCH_SPACE = {
    "learning_rate": [
        2e-4,
        3e-4,
        4e-4,
        5e-4,
    ],
}

FIXED_PARAMETERS = {
    "weight_decay": BEST_WEIGHT_DECAY,
    "scheduler": "none",

    "positive_weight": BEST_POSITIVE_WEIGHT,
    "bce_weight": BEST_BCE_WEIGHT,
    "dice_weight": BEST_DICE_WEIGHT,

    "batch_size": FULL_BATCH_SIZE,
    "image_size": config["data"]["image_size"],
    "num_workers": config["data"]["num_workers"],
    "seed": 42,
    "precision": "fp32",

    "model_parameters": BEST_MODEL_PARAMETERS,
}

SEARCH_NAME = (
    "bit_full_stage5_lr_refinement_"
    "40ep_seed42_v1"
)

stage5_seed42_results = run_parameter_search(
    search_space=SEARCH_SPACE,
    fixed_parameters=FIXED_PARAMETERS,
    model_name=config["model"]["name"],
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,
    number_of_epochs=HPO_EPOCHS,
    output_root=HPO_OUTPUT_ROOT,
    search_name=SEARCH_NAME,
    gpu_ids=HPO_GPU_IDS,
    checkpoint_interval=5,
)

TOP_LEARNING_RATES = [
    result["parameters"]["learning_rate"]
    for result in sorted(
        stage5_seed42_results,
        key=lambda result: result["best_f1"],
        reverse=True,
    )[:2]
]

print(
    "TOP_LEARNING_RATES:",
    TOP_LEARNING_RATES,
)

Trials: 4
GPUs: [0, 1, 2, 3]

[START] Trial 000 -> GPU 0
seed=42 | lr=0.0002 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=40

[START] Trial 001 -> GPU 1
seed=42 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=40

[START] Trial 002 -> GPU 2
seed=42 | lr=0.0004 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8

In [26]:
# ============================================================
# STAGE 5B — LEARNING RATE MULTI-SEED
# ============================================================

SEARCH_SPACE = {
    "learning_rate": TOP_LEARNING_RATES,
    "seed": [
        42,
        123,
        777,
    ],
}

FIXED_PARAMETERS = {
    "weight_decay": BEST_WEIGHT_DECAY,
    "scheduler": "none",

    "positive_weight": BEST_POSITIVE_WEIGHT,
    "bce_weight": BEST_BCE_WEIGHT,
    "dice_weight": BEST_DICE_WEIGHT,

    "batch_size": FULL_BATCH_SIZE,
    "image_size": config["data"]["image_size"],
    "num_workers": config["data"]["num_workers"],
    "precision": "fp32",

    "model_parameters": BEST_MODEL_PARAMETERS,
}

SEARCH_NAME = (
    "bit_full_stage5_lr_"
    "multiseed_60ep_v3"
)

stage5_multiseed_results = run_parameter_search(
    search_space=SEARCH_SPACE,
    fixed_parameters=FIXED_PARAMETERS,
    model_name=config["model"]["name"],
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,
    number_of_epochs=VERIFY_EPOCHS,
    output_root=HPO_OUTPUT_ROOT,
    search_name=SEARCH_NAME,
    gpu_ids=HPO_GPU_IDS,
    checkpoint_interval=5,
)


learning_rate_scores = {
    learning_rate: []
    for learning_rate in TOP_LEARNING_RATES
}

for result in stage5_multiseed_results:
    learning_rate = (
        result["parameters"]["learning_rate"]
    )

    learning_rate_scores[
        learning_rate
    ].append(
        result["best_f1"]
    )


for learning_rate, scores in (
    learning_rate_scores.items()
):
    print(
        learning_rate,
        "mean F1:",
        sum(scores) / len(scores),
        "scores:",
        scores,
    )


BEST_LEARNING_RATE = max(
    learning_rate_scores,
    key=lambda learning_rate: (
        sum(learning_rate_scores[learning_rate])
        / len(learning_rate_scores[learning_rate])
    ),
)

print(
    "BEST_LEARNING_RATE:",
    BEST_LEARNING_RATE,
)

Trials: 6
GPUs: [0, 1, 2, 3]

[START] Trial 000 -> GPU 0
seed=42 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=60

[START] Trial 001 -> GPU 1
seed=123 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=60

[START] Trial 002 -> GPU 2
seed=777 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth

In [28]:
# ============================================================
# STAGE 6A — FINAL BATCH × LR SCREENING
# ============================================================

SEARCH_SPACE = {
    "batch_size": [
        4,
        8,
    ],
    "learning_rate": TOP_LEARNING_RATES,
}

FIXED_PARAMETERS = {
    "weight_decay": BEST_WEIGHT_DECAY,
    "scheduler": "none",

    "positive_weight": BEST_POSITIVE_WEIGHT,
    "bce_weight": BEST_BCE_WEIGHT,
    "dice_weight": BEST_DICE_WEIGHT,

    "image_size": config["data"]["image_size"],
    "num_workers": config["data"]["num_workers"],
    "seed": 42,
    "precision": "fp32",

    "model_parameters": BEST_MODEL_PARAMETERS,
}

SEARCH_NAME = (
    "bit_full_stage6_batch_lr_joint_"
    "40ep_seed42_v2"
)

stage6_seed42_results = run_parameter_search(
    search_space=SEARCH_SPACE,
    fixed_parameters=FIXED_PARAMETERS,
    model_name=config["model"]["name"],
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,
    number_of_epochs=HPO_EPOCHS,
    output_root=HPO_OUTPUT_ROOT,
    search_name=SEARCH_NAME,
    gpu_ids=HPO_GPU_IDS,
    checkpoint_interval=5,
)

TOP_FINAL_CANDIDATES = sorted(
    stage6_seed42_results,
    key=lambda result: result["best_f1"],
    reverse=True,
)[:2]

print("TOP_FINAL_CANDIDATES:")

for result in TOP_FINAL_CANDIDATES:
    print(
        "batch:",
        result["parameters"]["batch_size"],
        "| lr:",
        result["parameters"]["learning_rate"],
        "| F1:",
        result["best_f1"],
    )

Trials: 4
GPUs: [0, 1, 2, 3]

[START] Trial 000 -> GPU 0
seed=42 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=40

[START] Trial 001 -> GPU 1
seed=42 | lr=0.0004 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=40

[START] Trial 002 -> GPU 2
seed=42 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=8 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 GiB. GPU 2 has a total capacity of 79.25 GiB of which 341.50 MiB is free. Including non-PyTorch memory, this process has 61.92 GiB memory in use. Of the allocated memory 61.05 GiB is allocated by PyTorch, and 364.52 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [29]:
# ============================================================
# STAGE 6 — RERUN MISSING TRIAL
# batch=8, lr=3e-4
# ============================================================

SEARCH_SPACE = {
    "batch_size": [
        8,
    ],
    "learning_rate": [
        3e-4,
    ],
}

FIXED_PARAMETERS = {
    "weight_decay": BEST_WEIGHT_DECAY,
    "scheduler": "none",

    "positive_weight": BEST_POSITIVE_WEIGHT,
    "bce_weight": BEST_BCE_WEIGHT,
    "dice_weight": BEST_DICE_WEIGHT,

    "image_size": config["data"]["image_size"],
    "num_workers": config["data"]["num_workers"],
    "seed": 42,
    "precision": "fp32",

    "model_parameters": BEST_MODEL_PARAMETERS,
}

SEARCH_NAME = (
    "bit_full_stage6_missing_"
    "batch8_lr3e4_40ep_seed42_v1"
)

stage6_missing_results = run_parameter_search(
    search_space=SEARCH_SPACE,
    fixed_parameters=FIXED_PARAMETERS,
    model_name=config["model"]["name"],
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,
    number_of_epochs=HPO_EPOCHS,
    output_root=HPO_OUTPUT_ROOT,
    search_name=SEARCH_NAME,
    gpu_ids=[3],
    checkpoint_interval=5,
)

Trials: 1
GPUs: [3]

[START] Trial 000 -> GPU 3
seed=42 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=8 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=40
[Trial 000 | GPU 3] deterministic_algorithms=True | cudnn_deterministic=True | cudnn_benchmark=False

[Trial 000 | GPU 3] Epoch 1/40
Train loss:       0.757196
Validation loss:  0.749511
Best threshold:   0.85
Precision:        0.370895
Recall:           0.397938
F1:               0.383940
IoU:              0.237578

[Trial 000 | GPU 3] Epoch 2/40
Train loss:       0.604135
Validation loss:  0.914009
Best threshold:   0.85
Precision:        0.278605
Recall:           0.329774
F1:               0.302038
IoU:              0.177882

[Trial 000 | GPU 3] Epoch 3/40
Train loss:       0.538590
Validation loss:  0.656577
Best thresho

In [30]:
# ============================================================
# REBUILD COMPLETE STAGE 6 RESULTS
# ============================================================

stage6_seed42_results = [
    {
        "parameters": {
            "batch_size": 4,
            "learning_rate": 3e-4,
        },
        "best_f1": 0.831951,
        "best_iou": 0.712257,
    },
    {
        "parameters": {
            "batch_size": 4,
            "learning_rate": 4e-4,
        },
        "best_f1": 0.819805,
        "best_iou": 0.694635,
    },
    {
        "parameters": {
            "batch_size": 8,
            "learning_rate": 4e-4,
        },
        "best_f1": 0.786021,
        "best_iou": 0.647474,
    },
    *stage6_missing_results,
]


TOP_FINAL_CANDIDATES = sorted(
    stage6_seed42_results,
    key=lambda result: result["best_f1"],
    reverse=True,
)[:2]


print("TOP_FINAL_CANDIDATES:")

for result in TOP_FINAL_CANDIDATES:
    print(
        "batch:",
        result["parameters"]["batch_size"],
        "| lr:",
        result["parameters"]["learning_rate"],
        "| F1:",
        result["best_f1"],
    )

TOP_FINAL_CANDIDATES:
batch: 4 | lr: 0.0003 | F1: 0.831951
batch: 4 | lr: 0.0004 | F1: 0.819805


In [31]:
final_candidate_summary = []


for index, seed42_result in enumerate(
    TOP_FINAL_CANDIDATES,
    start=1,
):
    candidate_batch_size = (
        seed42_result["parameters"]["batch_size"]
    )

    candidate_learning_rate = (
        seed42_result["parameters"]["learning_rate"]
    )

    SEARCH_SPACE = {
        "seed": [
            42,
            123,
            777,
        ],
    }

    FIXED_PARAMETERS = {
        "learning_rate": candidate_learning_rate,
        "weight_decay": BEST_WEIGHT_DECAY,
        "scheduler": "none",

        "positive_weight": BEST_POSITIVE_WEIGHT,
        "bce_weight": BEST_BCE_WEIGHT,
        "dice_weight": BEST_DICE_WEIGHT,

        "batch_size": candidate_batch_size,
        "image_size": config["data"]["image_size"],
        "num_workers": config["data"]["num_workers"],
        "precision": "fp32",

        "model_parameters": BEST_MODEL_PARAMETERS,
    }

    SEARCH_NAME = (
        f"bit_full_stage6_candidate"
        f"{index}_multiseed_60ep_v2"
    )

    candidate_results = run_parameter_search(
        search_space=SEARCH_SPACE,
        fixed_parameters=FIXED_PARAMETERS,
        model_name=config["model"]["name"],
        dataset_root=dataset_root,
        thresholds=THRESHOLDS,
        number_of_epochs=VERIFY_EPOCHS,
        output_root=HPO_OUTPUT_ROOT,
        search_name=SEARCH_NAME,
        gpu_ids=HPO_GPU_IDS,
        checkpoint_interval=5,
    )

    f1_scores = [
        result["best_f1"]
        for result in candidate_results
    ]

    mean_f1 = (
        sum(f1_scores)
        / len(f1_scores)
    )

    final_candidate_summary.append(
        {
            "batch_size": candidate_batch_size,
            "learning_rate": candidate_learning_rate,
            "mean_f1": mean_f1,
            "scores": f1_scores,
        }
    )

    print(
        "batch:",
        candidate_batch_size,
        "lr:",
        candidate_learning_rate,
        "mean F1:",
        mean_f1,
        "scores:",
        f1_scores,
    )


BEST_FINAL_CANDIDATE = max(
    final_candidate_summary,
    key=lambda candidate: candidate["mean_f1"],
)

BEST_BATCH_SIZE = (
    BEST_FINAL_CANDIDATE["batch_size"]
)

BEST_LEARNING_RATE = (
    BEST_FINAL_CANDIDATE["learning_rate"]
)

print()
print(
    "BEST_BATCH_SIZE:",
    BEST_BATCH_SIZE,
)

print(
    "BEST_LEARNING_RATE:",
    BEST_LEARNING_RATE,
)

Trials: 3
GPUs: [0, 1, 2, 3]

[START] Trial 000 -> GPU 0
seed=42 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=60

[START] Trial 001 -> GPU 1
seed=123 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=60

[START] Trial 002 -> GPU 2
seed=777 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth

In [32]:
# ============================================================
# FINAL MEDIUM FULL — 100 EPOCHS × 3 SEEDS
# ============================================================

SEARCH_SPACE = {
    "seed": [
        42,
        123,
        777,
    ],
}

FIXED_PARAMETERS = {
    "learning_rate": BEST_LEARNING_RATE,
    "weight_decay": BEST_WEIGHT_DECAY,
    "scheduler": "none",

    "positive_weight": BEST_POSITIVE_WEIGHT,
    "bce_weight": BEST_BCE_WEIGHT,
    "dice_weight": BEST_DICE_WEIGHT,

    "batch_size": BEST_BATCH_SIZE,
    "image_size": config["data"]["image_size"],
    "num_workers": config["data"]["num_workers"],
    "precision": "fp32",

    "model_parameters": BEST_MODEL_PARAMETERS,
}

SEARCH_NAME = (
    "bit_medium_full_final_"
    "100ep_seeds42_123_777_v1"
)

final_results = run_parameter_search(
    search_space=SEARCH_SPACE,
    fixed_parameters=FIXED_PARAMETERS,
    model_name=config["model"]["name"],
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,
    number_of_epochs=FINAL_EPOCHS,
    output_root=HPO_OUTPUT_ROOT,
    search_name=SEARCH_NAME,
    gpu_ids=HPO_GPU_IDS,
    checkpoint_interval=5,
)

Trials: 3
GPUs: [0, 1, 2, 3]

[START] Trial 000 -> GPU 0
seed=42 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=100

[START] Trial 001 -> GPU 1
seed=123 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=100

[START] Trial 002 -> GPU 2
seed=777 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_dep

OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 74.94 MiB is free. Process 1070 has 1.27 GiB memory in use. Including non-PyTorch memory, this process has 1.23 GiB memory in use. Of the allocated memory 704.46 MiB is allocated by PyTorch, and 63.54 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [33]:
# ============================================================
# FINAL MEDIUM FULL — RERUN MISSING SEED 42
# ============================================================

HPO_OUTPUT_ROOT = (
    project_root
    / "artifacts"
    / "hyperparameter_search"
)

MODEL_PARAMETERS = {
    "in_channels": 3,
    "out_channels": 1,
    "embedding_dim": 64,
    "token_length": 4,
    "num_heads": 8,
    "encoder_head_dim": 64,
    "decoder_head_dim": 64,
    "encoder_depth": 1,
    "decoder_depth": 8,
    "dropout": 0.0,
    "pretrained_backbone": False,
}

SEARCH_SPACE = {
    "seed": [42],
}

FIXED_PARAMETERS = {
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "scheduler": "none",

    "positive_weight": 10,
    "bce_weight": 0.3,
    "dice_weight": 0.7,

    "batch_size": 4,
    "image_size": config["data"]["image_size"],
    "num_workers": config["data"]["num_workers"],
    "precision": "fp32",

    "model_parameters": MODEL_PARAMETERS,
}

SEARCH_NAME = (
    "bit_medium_full_final_"
    "100ep_seed42_rerun_v1"
)

seed42_final_results = run_parameter_search(
    search_space=SEARCH_SPACE,
    fixed_parameters=FIXED_PARAMETERS,
    model_name=config["model"]["name"],
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,
    number_of_epochs=100,
    output_root=HPO_OUTPUT_ROOT,
    search_name=SEARCH_NAME,

    # Один trial на отдельной GPU.
    # GPU 3 в предыдущем финальном запуске не использовалась.
    gpu_ids=[3],

    checkpoint_interval=5,
)

Trials: 1
GPUs: [3]

[START] Trial 000 -> GPU 3
seed=42 | lr=0.0003 | wd=0.0001 | pw=10 | BCE/Dice=0.3/0.7 | batch=4 | image_size=1024 | in_channels=3 | out_channels=1 | embedding_dim=64 | token_length=4 | num_heads=8 | encoder_head_dim=64 | decoder_head_dim=64 | encoder_depth=1 | decoder_depth=8 | dropout=0.0 | pretrained_backbone=False | scheduler=none
epochs=100
[Trial 000 | GPU 3] deterministic_algorithms=True | cudnn_deterministic=True | cudnn_benchmark=False

[Trial 000 | GPU 3] Epoch 1/100
Train loss:       0.741705
Validation loss:  0.733776
Best threshold:   0.70
Precision:        0.483081
Recall:           0.440631
F1:               0.460881
IoU:              0.299445

[Trial 000 | GPU 3] Epoch 2/100
Train loss:       0.589990
Validation loss:  0.734815
Best threshold:   0.50
Precision:        0.513711
Recall:           0.478601
F1:               0.495535
IoU:              0.329376

[Trial 000 | GPU 3] Epoch 3/100
Train loss:       0.553405
Validation loss:  0.722964
Best thr

In [ ]:
scheduler_name = parameters.get(
    "scheduler",
    "none",
)

if scheduler_name == "none":
    scheduler = None

elif scheduler_name == "drop40":
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[40],
        gamma=1 / 3,
    )

elif scheduler_name == "drop60":
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[60],
        gamma=1 / 3,
    )

else:
    raise ValueError(
        f"Unknown scheduler: {scheduler_name}"
    )


# ============================================================
# LR DROP EXPERIMENT
# 3e-4 -> 1e-4 after epoch 40 or 60
# Total: 80 epochs
# ============================================================

LR_SCHEDULE_EPOCHS = 80

SEARCH_SPACE = {
    "scheduler": [
        "drop40",
        "drop60",
    ],
    "seed": [
        42,
        123,
        777,
    ],
}

FIXED_PARAMETERS = {
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,

    "positive_weight": 10,
    "bce_weight": 0.3,
    "dice_weight": 0.7,

    "batch_size": 4,
    "image_size": config["data"]["image_size"],
    "num_workers": config["data"]["num_workers"],
    "precision": "fp32",

    "model_parameters": {
        "in_channels": 3,
        "out_channels": 1,
        "embedding_dim": 64,
        "token_length": 4,
        "num_heads": 8,
        "encoder_head_dim": 64,
        "decoder_head_dim": 64,
        "encoder_depth": 1,
        "decoder_depth": 8,
        "dropout": 0.0,
        "pretrained_backbone": False,
    },
}

SEARCH_NAME = (
    "bit_full_lr_schedule_"
    "80ep_multiseed_v1"
)

lr_schedule_results = run_parameter_search(
    search_space=SEARCH_SPACE,
    fixed_parameters=FIXED_PARAMETERS,
    model_name=config["model"]["name"],
    dataset_root=dataset_root,
    thresholds=THRESHOLDS,
    number_of_epochs=LR_SCHEDULE_EPOCHS,
    output_root=HPO_OUTPUT_ROOT,
    search_name=SEARCH_NAME,
    gpu_ids=HPO_GPU_IDS,
    checkpoint_interval=5,
)

In [10]:
# ============================================================
# SAVE LR SCHEDULE EXPERIMENT SUMMARY
# ============================================================

LR_EXPERIMENT_DIRECTORY = (
    HPO_OUTPUT_ROOT
    / SEARCH_NAME
)

LR_EXPERIMENT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

LR_EXPERIMENT_SUMMARY = {
    "experiment": (
        "learning_rate_drop"
    ),

    "total_epochs": 80,

    "initial_learning_rate": 3e-4,
    "reduced_learning_rate": 1e-4,

    "tested_schedules": {
        "none": (
            "3e-4 for epochs 1-80"
        ),
        "drop40": (
            "3e-4 for epochs 1-40, "
            "1e-4 for epochs 41-80"
        ),
        "drop60": (
            "3e-4 for epochs 1-60, "
            "1e-4 for epochs 61-80"
        ),
    },

    "seeds": [
        42,
        123,
        777,
    ],

    "comparison": comparison,

    "best_schedule": BEST_SCHEDULE,
}

SUMMARY_PATH = (
    LR_EXPERIMENT_DIRECTORY
    / "summary.json"
)

with SUMMARY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        LR_EXPERIMENT_SUMMARY,
        file,
        indent=2,
    )

print(
    "Saved:",
    SUMMARY_PATH,
)

NameError: name 'SEARCH_NAME' is not defined

In [11]:
# ============================================================
# SAVE MEDIUM FULL FINAL PARAMETERS
# ============================================================

MEDIUM_FULL_DIRECTORY = (
    project_root
    / "artifacts"
    / "final_models"
    / "medium_full"
)

MEDIUM_FULL_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


MEDIUM_FULL_PARAMETERS = {
    "model": "bit",

    "model_parameters": {
        "in_channels": 3,
        "out_channels": 1,
        "embedding_dim": 64,
        "token_length": 4,
        "num_heads": 8,
        "encoder_head_dim": 64,
        "decoder_head_dim": 64,
        "encoder_depth": 1,
        "decoder_depth": 8,
        "dropout": 0.0,
        "pretrained_backbone": False,
    },

    "parameter_count": 4301317,

    "data": {
        "image_size": config["data"]["image_size"],
        "batch_size": 4,
        "num_workers": config["data"]["num_workers"],
    },

    "training": {
        "epochs": 100,
        "learning_rate": 3e-4,
        "weight_decay": 1e-4,
        "scheduler": "none",
        "precision": "fp32",

        "loss": {
            "positive_weight": 10,
            "bce_weight": 0.3,
            "dice_weight": 0.7,
        },

        "seeds": [
            42,
            123,
            777,
        ],
    },
}


FINAL_PARAMETERS_PATH = (
    MEDIUM_FULL_DIRECTORY
    / "final_parameters.json"
)

with FINAL_PARAMETERS_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        MEDIUM_FULL_PARAMETERS,
        file,
        indent=2,
    )

print(
    "Saved:",
    FINAL_PARAMETERS_PATH,
)

Saved: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/final_models/medium_full/final_parameters.json


In [12]:
# ============================================================
# SAVE MEDIUM FULL VALIDATION BENCHMARK
# ============================================================

MEDIUM_FULL_BENCHMARK = {
    "model": "bit",
    "variant": "medium_full",
    "split": "validation",

    "seeds": {
        "42": {
            "best_epoch": 81,
            "threshold": 0.50,
            "precision": 0.859092,
            "recall": 0.861373,
            "f1": 0.860231,
            "iou": 0.754741,
        },

        "123": {
            "best_epoch": 100,
            "threshold": 0.70,
            "precision": 0.865836,
            "recall": 0.856103,
            "f1": 0.860942,
            "iou": 0.755837,
        },

        "777": {
            "best_epoch": 82,
            "threshold": 0.45,
            "precision": 0.851853,
            "recall": 0.845741,
            "f1": 0.848786,
            "iou": 0.737296,
        },
    },

    "mean": {
        "precision": 0.858927,
        "recall": 0.854406,
        "f1": 0.856653,
        "iou": 0.749291,
    },

    "std": {
        "precision": 0.005710,
        "recall": 0.006494,
        "f1": 0.005570,
        "iou": 0.008494,
    },

    "std_type": "population",
}


BENCHMARK_PATH = (
    MEDIUM_FULL_DIRECTORY
    / "benchmark.json"
)

with BENCHMARK_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        MEDIUM_FULL_BENCHMARK,
        file,
        indent=2,
    )

print(
    "Saved:",
    BENCHMARK_PATH,
)

Saved: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/final_models/medium_full/benchmark.json
